In [ ]:
def chunk_document(
    document,
    tokenizer,
    target_tokens=360,
    max_tokens=450,
    overlap_tokens=50,
):
    """
    Convert a raw document into retrieval-ready chunks.

    The function accepts the DOCUMENT itself and is responsible for:

    1. Cleaning PDF/extraction artifacts.
    2. Identifying major document sections.
    3. Preserving semantic boundaries such as paragraphs and bullets.
    4. Combining smaller pieces until the target token size is reached.
    5. Preventing chunks from exceeding max_tokens.
    6. Adding semantic overlap between neighboring chunks.
    7. Returning a list of chunks.

    The goal is NOT to create equally sized pieces of text.

    Instead, each chunk should be a reasonably self-contained piece of
    information that can be retrieved independently by an embedding model.
    """

    # ------------------------------------------------------------
    # Configuration validation
    # ------------------------------------------------------------

    if not document or not document.strip():
        return []

    if not 0 <= overlap_tokens < target_tokens <= max_tokens:
        raise ValueError(
            "Require 0 <= overlap_tokens < target_tokens <= max_tokens."
        )

    # ------------------------------------------------------------
    # Helper: count tokens
    #
    # Token counts should be based on the SAME tokenizer that will
    # ultimately be used to create embeddings/prompts.
    # ------------------------------------------------------------

    def token_count(text):
        return len(tokenizer.encode(text))

    # ------------------------------------------------------------
    # Step 1: Clean the raw document
    #
    # PDF extraction frequently introduces:
    #
    # - repeated headers/footers
    # - excessive whitespace
    # - broken lines
    # - page artifacts
    #
    # These should be removed BEFORE chunking.
    # ------------------------------------------------------------

    lines = [
        line.strip()
        for line in document.splitlines()
        if line.strip()
    ]

    cleaned_lines = []

    for line in lines:

        # Remove the repeated Netflix PDF footer.
        if line.startswith("© 2024 Netflix, Inc."):
            continue

        # Remove obvious page-number-only lines.
        if line.isdigit():
            continue

        cleaned_lines.append(line)

    # ------------------------------------------------------------
    # Step 2: Identify structural boundaries
    #
    # ALL-CAPS headings are treated as hard boundaries.
    #
    # For this document this preserves sections such as:
    #
    # THE DREAM TEAM
    # PEOPLE OVER PROCESS
    # UNCOMFORTABLY EXCITING
    # ARTISTIC EXPRESSION
    # GREAT AND ALWAYS BETTER
    #
    # We don't want a chunk to casually span unrelated sections.
    # ------------------------------------------------------------

    sections = []
    current_section = []
    current_heading = None

    for line in cleaned_lines:

        is_heading = (
            line.isupper()
            and len(line.split()) <= 8
            and not line.endswith(".")
        )

        if is_heading:

            if current_section:
                sections.append({
                    "heading": current_heading,
                    "lines": current_section,
                })

            current_heading = line
            current_section = []

        else:
            current_section.append(line)

    if current_section:
        sections.append({
            "heading": current_heading,
            "lines": current_section,
        })

    # ------------------------------------------------------------
    # Step 3: Convert each section into semantic units
    #
    # We want to preserve:
    #
    # - paragraphs
    # - bullet points
    # - numbered items
    #
    # We do NOT want to blindly split every N characters.
    # ------------------------------------------------------------

    semantic_units = []

    for section in sections:

        heading = section["heading"]
        lines = section["lines"]

        current_paragraph = []

        for line in lines:

            # Detect bullets.
            is_bullet = (
                line.startswith(("•", "-", "*", "–", "—"))
                or line[:2].isdigit() and line[2:3] in {".", ")"}
            )

            if is_bullet:

                # Finish any paragraph before starting the bullet.
                if current_paragraph:
                    semantic_units.append({
                        "section": heading,
                        "text": " ".join(current_paragraph),
                    })
                    current_paragraph = []

                semantic_units.append({
                    "section": heading,
                    "text": line,
                })

            else:
                current_paragraph.append(line)

        # Flush final paragraph.
        if current_paragraph:
            semantic_units.append({
                "section": heading,
                "text": " ".join(current_paragraph),
            })

    # ------------------------------------------------------------
    # Step 4: Split oversized semantic units
    #
    # Normally we preserve semantic units intact.
    #
    # However, occasionally a single paragraph can be larger than
    # max_tokens. We have no choice but to split it.
    #
    # This is a FALLBACK, not the normal chunking strategy.
    # ------------------------------------------------------------

    normalized_units = []

    for unit in semantic_units:

        text = unit["text"]

        if token_count(text) <= max_tokens:
            normalized_units.append(unit)
            continue

        tokens = tokenizer.encode(text)

        for start in range(0, len(tokens), max_tokens):

            piece = tokenizer.decode(
                tokens[start:start + max_tokens]
            ).strip()

            normalized_units.append({
                "section": unit["section"],
                "text": piece,
            })

    # ------------------------------------------------------------
    # Step 5: Build chunks
    #
    # We greedily add semantic units until adding another unit
    # would exceed the target size.
    #
    # target_tokens is the desired size.
    # max_tokens is the absolute ceiling.
    #
    # This distinction is important:
    #
    # target = "I'd like chunks around this size."
    # max    = "Never intentionally go beyond this size."
    # ------------------------------------------------------------

    chunks = []

    i = 0

    while i < len(normalized_units):

        chunk_units = []
        chunk_tokens = 0

        starting_index = i

        while i < len(normalized_units):

            unit = normalized_units[i]

            unit_tokens = token_count(unit["text"])

            # Don't add another semantic unit if doing so would
            # take the chunk beyond our preferred target.
            #
            # We allow the FIRST unit even if it is unusually large;
            # oversized units were already handled above.
            if (
                chunk_units
                and chunk_tokens + unit_tokens > target_tokens
            ):
                break

            # Absolute safety check.
            if (
                chunk_units
                and chunk_tokens + unit_tokens > max_tokens
            ):
                break

            chunk_units.append(unit)
            chunk_tokens += unit_tokens
            i += 1

        # --------------------------------------------------------
        # Safety fallback.
        #
        # This should almost never happen because oversized units
        # were normalized above.
        # --------------------------------------------------------

        if not chunk_units:
            chunk_units.append(normalized_units[i])
            i += 1

        # --------------------------------------------------------
        # Build the actual chunk text.
        #
        # Include the section heading because it provides valuable
        # context when the chunk is retrieved independently.
        # --------------------------------------------------------

        section = chunk_units[0]["section"]

        body = "\n\n".join(
            unit["text"]
            for unit in chunk_units
        )

        if section:
            chunk_text = f"{section}\n\n{body}"
        else:
            chunk_text = body

        chunks.append({
            "text": chunk_text,
            "section": section,
            "token_count": token_count(chunk_text),
        })

        # --------------------------------------------------------
        # Step 6: Semantic overlap
        #
        # Do NOT overlap arbitrary token ranges.
        #
        # Instead, carry complete semantic units from the end of
        # the previous chunk into the next chunk.
        #
        # This keeps the overlap meaningful.
        # --------------------------------------------------------

        if overlap_tokens > 0 and i < len(normalized_units):

            overlap_units = []
            overlap_size = 0

            for unit in reversed(chunk_units):

                unit_tokens = token_count(unit["text"])

                if (
                    overlap_units
                    and overlap_size + unit_tokens > overlap_tokens
                ):
                    break

                overlap_units.insert(0, unit)
                overlap_size += unit_tokens

                if overlap_size >= overlap_tokens:
                    break

            # Move the next chunk's starting position backwards
            # so the selected semantic units are included again.
            overlap_start = i - len(overlap_units)

            # Never move backwards past the beginning of the
            # current chunk. This prevents an infinite loop.
            i = max(
                starting_index + 1,
                overlap_start,
            )

    return chunks
